# ISBP Non-Regular Instances — Exp 1 & Exp 2 (w̄ ∈ {4,...,15})

Non-regular instances: bin capacities $u_j$, bin costs $c_j$, and player weights $w_i$
are drawn randomly around their mean values with spread factor $\delta = 0.2$.

- $w_i \sim \text{UniformInt}[\lceil \bar w(1-\delta) \rceil,\; \lfloor \bar w(1+\delta) \rfloor]$
- $u_j \sim \text{UniformInt}[\lceil \bar u(1-\delta) \rceil,\; \lfloor \bar u(1+\delta) \rfloor]$
- $c_j \sim \text{UniformInt}[\lceil \bar u(1-\delta) \rceil,\; \lfloor \bar u(1+\delta) \rfloor]$

Feasibility enforced: $\sum_j u_j \ge \sum_i w_i$.
If violated after drawing, the smallest-capacity bins are bumped up.

**Exp 1**: Best PNE and POS (BRD + ZR @ alpha=1, weaker VEST cuts for irregular instances)
**Exp 2**: TAS for non-PNE instances

> **Note**: Exp 1B (weaker VEST cut off) is in a separate notebook `03b_run_isbp_nonregular_exp1b.ipynb`
> to ensure independent execution environment (eliminates memory/ordering bias).

In [ ]:
from pathlib import Path
import json
import time
import csv
import math
import traceback

import numpy as np
import pandas as pd

from gipg.isbp.instance import ISBPInstance
from gipg.isbp.heuristics import (
    brd_random_restart, alpha_of_profile,
    is_regular_instance,
)
from gipg.isbp.gzr import solve_gzr
from gipg.isbp.abs_search import solve_abs
from gipg.isbp.objectives import profile_costs
from gipg.isbp.social_optimum import solve_social_optimum, compute_pos

OUTDIR = Path('outputs')
OUTDIR.mkdir(exist_ok=True)
print('Imports OK.')

## 1. Non-Regular Instance Generation

In [ ]:
def generate_nonregular_instance(
    n_players: int,
    m_bins: int,
    w_bar: int,
    u_bar: int,
    delta: float = 0.2,
    seed: int = 0,
) -> ISBPInstance:
    """
    Generate a non-regular ISBP instance with heterogeneous parameters.

    - w_i ~ UniformInt[ceil(w_bar*(1-delta)), floor(w_bar*(1+delta))]
    - u_j ~ UniformInt[ceil(u_bar*(1-delta)), floor(u_bar*(1+delta))]
    - c_j ~ UniformInt[ceil(u_bar*(1-delta)), floor(u_bar*(1+delta))]  (centered on u_bar)

    Feasibility: sum(u_j) >= sum(w_i).  If violated, bump smallest capacities.
    """
    rng = np.random.default_rng(seed)
    players = list(range(n_players))
    bins = list(range(m_bins))

    # --- Compute ranges ---
    w_lo = max(1, math.ceil(w_bar * (1 - delta)))
    w_hi = math.floor(w_bar * (1 + delta))
    u_lo = max(1, math.ceil(u_bar * (1 - delta)))
    u_hi = math.floor(u_bar * (1 + delta))
    c_lo = max(1, math.ceil(u_bar * (1 - delta)))  # centered on u_bar
    c_hi = math.floor(u_bar * (1 + delta))

    # --- Draw random values (inclusive on both ends) ---
    weights = {i: int(rng.integers(w_lo, w_hi + 1)) for i in players}
    capacities = {j: int(rng.integers(u_lo, u_hi + 1)) for j in bins}
    costs = {j: float(rng.integers(c_lo, c_hi + 1)) for j in bins}

    # --- Enforce feasibility: sum(u_j) >= sum(w_i) ---
    total_w = sum(weights.values())
    total_u = sum(capacities.values())
    if total_u < total_w:
        deficit = total_w - total_u
        # Sort bins by capacity (ascending) and bump smallest ones
        sorted_bins = sorted(bins, key=lambda j: capacities[j])
        idx = 0
        while deficit > 0:
            j = sorted_bins[idx % m_bins]
            capacities[j] += 1
            deficit -= 1
            idx += 1

    # --- Build auxiliary sets T and S ---
    T = {j: list(range(0, capacities[j] + 1)) for j in bins}
    S = {
        i: {
            j: {
                t: list(range(0, min(t, weights[i]) + 1))
                for t in T[j]
            }
            for j in bins
        }
        for i in players
    }
    return ISBPInstance(players, bins, costs, capacities, weights, T, S)


# Quick test
test_inst = generate_nonregular_instance(5, 4, w_bar=8, u_bar=12, delta=0.2, seed=0)
print('Test instance:')
print(f'  weights:    {dict(test_inst.weights)}')
print(f'  capacities: {dict(test_inst.capacities)}')
print(f'  costs:      {dict(test_inst.costs)}')
print(f'  sum(w)={sum(test_inst.weights.values())}, sum(u)={sum(test_inst.capacities.values())}')
print(f'  regular?    {is_regular_instance(test_inst)}')

In [ ]:
def enumerate_tuples_regular(n_range, m_range, w_range):
    """
    Enumerate base (n, m, w_bar, u_bar) tuples.
    Same logic as regular case — these define the *mean* parameters.
    """
    tuples = []
    for n in n_range:
        for m in range(n + m_range[0], n + m_range[1]):
            if m <= 0:
                continue
            for w in w_range:
                nw = n * w
                q, r = divmod(nw, m)

                if r == 0:
                    u_candidates = [q, q+1, q+2, q+3]
                else:
                    u_candidates = [q+1, q+2, q+3, q+4]

                for u in u_candidates:
                    if (m*u - nw >= 0) and (m <= math.ceil(nw/u)):
                        tuples.append((n, m, w, u))
    return tuples

n_range = range(4, 11)
m_range = [-2, 4]     # m = n-2,...,n+3
w_range = range(4, 16) # w_bar = 4,...,15
DELTA = 0.2

tuples = enumerate_tuples_regular(n_range, m_range, w_range)
print(f'Base tuples: {len(tuples)}')
print(f'Delta: {DELTA}')
print(f'Total non-regular instances: {len(tuples)}')

In [ ]:
# --- Save the non-regular instance dataset ---
SEEDS = [0]
DELTA = 0.2

dataset_rows = []
for (n, m, w_bar, u_bar) in tuples:
    for seed in SEEDS:
        inst = generate_nonregular_instance(
            n_players=n, m_bins=m,
            w_bar=w_bar, u_bar=u_bar,
            delta=DELTA, seed=seed,
        )
        regular = is_regular_instance(inst)
        sum_w = sum(inst.weights.values())
        sum_u = sum(inst.capacities.values())

        dataset_rows.append({
            'seed': seed, 'n': n, 'm': m,
            'w_bar': w_bar, 'u_bar': u_bar, 'delta': DELTA,
            'regular': regular,
            'weights': str(dict(inst.weights)),
            'capacities': str(dict(inst.capacities)),
            'costs': str(dict(inst.costs)),
            'sum_w': sum_w, 'sum_u': sum_u,
            'feasible': sum_u >= sum_w,
        })

df_dataset = pd.DataFrame(dataset_rows)
dataset_path = OUTDIR / 'isbp_nonreg_instances.csv'
df_dataset.to_csv(dataset_path, index=False)

print(f'Saved {len(df_dataset)} instances to {dataset_path}')
print(f'Regular: {df_dataset["regular"].sum()}, Non-regular: {(~df_dataset["regular"]).sum()}')
print(f'All feasible: {df_dataset["feasible"].all()}')
print(df_dataset.head())

In [ ]:
# Controls
TOTAL_TIME_LIMIT = 600.0
BR_TIME_LIMIT = 10.0
SO_TIME_LIMIT = 600.0
TAS_TIME_LIMIT = 600.0
TAS_EPS = 1e-3

# Output CSV paths
exp1_path = OUTDIR / 'isbp_nonreg_exp1_brd_zr_pos.csv'
exp2_path = OUTDIR / 'isbp_nonreg_exp2_tas.csv'
combined_path = OUTDIR / 'isbp_nonreg_results_all.csv'
err_path = OUTDIR / 'isbp_nonreg_errors.csv'

# CSV append helper
def append_row(path: Path, header, rowdict):
    new_file = not path.exists()
    with path.open('a', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=header)
        if new_file:
            w.writeheader()
        w.writerow({k: rowdict.get(k, '') for k in header})

print(f'Tuples: {len(tuples)}, Seeds: {len(SEEDS)}, Total runs: {len(tuples)*len(SEEDS)}')

## 2. Experiment 1: Best PNE and POS (BRD + ZR @ alpha=1)

In [ ]:
exp1_header = [
    'seed', 'n', 'm', 'w_bar', 'u_bar', 'delta', 'regular',
    'sum_w', 'sum_u', 'feasibility_repairs',
    'brd_found_pne', 'brd_alpha', 'brd_time',
    'zr_status', 'zr_gurobi_status', 'zr_mip_gap', 'zr_obj_bound', 'zr_runtime', 'zr_time',
    'zr_eis_added_total', 'zr_eis_added_symmetric', 'zr_eis_added_core',
    'zr_first_pne_time',
    'best_pne_cost',
    'so_status', 'so_cost', 'so_time',
    'pos',
]

err_header = ['seed', 'n', 'm', 'w_bar', 'u_bar', 'delta', 'stage', 'error', 'traceback']

results_exp1 = []

total = 0
for (n, m, w_bar, u_bar) in tuples:
    for seed in SEEDS:
        total += 1
        tag = f'n{n}_m{m}_wb{w_bar}_ub{u_bar}_d{DELTA}_s{seed}'

        row = {
            'seed': seed, 'n': n, 'm': m,
            'w_bar': w_bar, 'u_bar': u_bar, 'delta': DELTA,
        }

        try:
            inst = generate_nonregular_instance(
                n_players=n, m_bins=m,
                w_bar=w_bar, u_bar=u_bar,
                delta=DELTA, seed=seed,
            )
            regular = is_regular_instance(inst)
            row['regular'] = regular

            sum_w = sum(inst.weights.values())
            sum_u = sum(inst.capacities.values())
            # How many capacity bumps were needed
            u_lo = max(1, math.ceil(u_bar * (1 - DELTA)))
            u_hi = math.floor(u_bar * (1 + DELTA))
            nominal_sum_u = sum(
                int(np.random.default_rng(seed).integers(u_lo, u_hi + 1))
                for _ in range(m)
            )  # approximate; exact repair count tracked below
            row['sum_w'] = sum_w
            row['sum_u'] = sum_u
            row['feasibility_repairs'] = max(0, sum_w - nominal_sum_u)

            # --- Phase 1: BRD ---
            x_brd, brd_pne, brd_time = brd_random_restart(
                inst, max_init=3, max_round=15,
                seed=seed,
            )
            brd_alpha = alpha_of_profile(inst, x_brd) if x_brd else float('inf')
            row['brd_found_pne'] = brd_pne
            row['brd_alpha'] = brd_alpha
            row['brd_time'] = brd_time

            # --- Phase 2: ZR @ alpha=1, warm-started with BRD profile ---
            zr_time_limit = max(TOTAL_TIME_LIMIT - brd_time, 60.0)
            warm = x_brd if brd_pne else None

            # BRD+GZR for irregular: Techniques #1, #2, #3 active
            zr = solve_gzr(
                inst,
                alpha=1.0,
                time_limit=zr_time_limit,
                log_to_console=0,
                stop_at_first_pne=False,
                warm_start=warm,
            )

            zr_status = zr.get('status')
            row['zr_gurobi_status'] = zr.get('gurobi_status')
            row['zr_mip_gap'] = zr.get('mip_gap')
            row['zr_obj_bound'] = zr.get('obj_bound')
            row['zr_runtime'] = zr.get('runtime')
            row['zr_time'] = brd_time + float(zr.get('runtime', 0))
            row['zr_eis_added_total'] = zr.get('eis_added_total', 0)
            row['zr_eis_added_symmetric'] = zr.get('eis_added_symmetric', 0)
            row['zr_eis_added_core'] = zr.get('eis_added_core', 0)
            row['zr_first_pne_time'] = (brd_time + zr['first_pne_time']
                                                    if zr.get('first_pne_time') is not None else None)

            # --- Post-ZR fallback: re-verify BRD profile if ZR was INTERRUPTED/UNKNOWN ---
            zr_profile = zr.get('x_profile')
            if zr_profile is None and zr_status in ('INTERRUPTED', 'UNKNOWN') and brd_pne:
                verify_alpha = alpha_of_profile(inst, x_brd)
                if verify_alpha <= 1.0 + 1e-9:
                    zr_status = 'OPTIMAL'
                    zr_profile = x_brd

            row['zr_status'] = zr_status

            # Best PNE cost
            if zr_profile is not None:
                _, total_cost = profile_costs(
                    players=inst.players, bins=inst.bins,
                    costs=inst.costs, x_profile=zr_profile,
                )
                row['best_pne_cost'] = total_cost
            elif brd_pne and x_brd is not None:
                _, total_cost = profile_costs(
                    players=inst.players, bins=inst.bins,
                    costs=inst.costs, x_profile=x_brd,
                )
                row['best_pne_cost'] = total_cost
            else:
                row['best_pne_cost'] = None

            # --- Phase 3: Social Optimum ---
            so_res = solve_social_optimum(inst, time_limit=SO_TIME_LIMIT, verbose=False)
            row['so_status'] = so_res.status
            row['so_cost'] = so_res.opt_cost
            row['so_time'] = so_res.runtime

            # POS = best_PNE_cost / SO_cost (minimization: POS >= 1)
            if row.get('best_pne_cost') is not None and so_res.opt_cost is not None and so_res.opt_cost > 0:
                row['pos'] = compute_pos(so_res.opt_cost, row['best_pne_cost'])
            else:
                row['pos'] = None

            results_exp1.append(row)
            append_row(exp1_path, exp1_header, row)

            # Progress
            if total % 50 == 0 or total == len(tuples) * len(SEEDS):
                pne_str = 'BRD-PNE' if brd_pne else 'no-PNE'
                zr_str = f'{row["zr_status"]}'
                zr_t = f'{row["zr_time"]:.1f}s' if row.get('zr_time') is not None else '?s'
                pos_str = f'POS={row["pos"]:.3f}' if row.get('pos') is not None else 'POS=N/A'
                fpne_str = f'1stPNE={row["zr_first_pne_time"]:.1f}s' if row.get('zr_first_pne_time') is not None else '1stPNE=N/A'
                print(f'[{total}] {tag}: {pne_str} | ZR {zr_str} ({zr_t}) | {pos_str} | {fpne_str}')

        except Exception as e:
            append_row(err_path, err_header, {
                'seed': seed, 'n': n, 'm': m, 'w_bar': w_bar, 'u_bar': u_bar,
                'delta': DELTA, 'stage': 'EXP1', 'error': repr(e),
                'traceback': traceback.format_exc(),
            })
            print(f'  !! EXP1 failed ({tag}): {e}')
            continue

df_exp1 = pd.DataFrame(results_exp1)
print(f'\nExp1 complete: {len(df_exp1)} rows saved to {exp1_path}')


In [ ]:
# Experiment 1 Summary
print('=== Experiment 1 Summary (Non-Regular) ===')

n_total = len(df_exp1)
n_brd_pne = df_exp1['brd_found_pne'].sum()
n_regular = df_exp1['regular'].sum()
n_zr_opt = (df_exp1['zr_status'] == 'OPTIMAL').sum()
n_zr_inf = (df_exp1['zr_status'] == 'INFEASIBLE').sum()
n_zr_tl = (df_exp1['zr_status'] == 'TIME_LIMIT').sum()
n_zr_int = (df_exp1['zr_status'] == 'INTERRUPTED').sum()
n_pos = df_exp1['pos'].notna().sum()

print(f'Total instances: {n_total}')
print(f'Actually regular (delta produced identical params): {n_regular}')
print(f'BRD found PNE: {n_brd_pne} ({n_brd_pne/n_total:.1%})')
print(f'ZR OPTIMAL: {n_zr_opt} ({n_zr_opt/n_total:.1%})')
print(f'ZR INFEASIBLE: {n_zr_inf} ({n_zr_inf/n_total:.1%})')
print(f'ZR TIME_LIMIT: {n_zr_tl} ({n_zr_tl/n_total:.1%})')
if n_zr_int > 0:
    print(f'ZR INTERRUPTED: {n_zr_int} ({n_zr_int/n_total:.1%})')
print(f'POS computed: {n_pos} ({n_pos/n_total:.1%})')
print(f'Feasibility repairs needed: {(df_exp1["feasibility_repairs"] > 0).sum()}')

# By (n, m)
print('\n--- By (n, m) ---')
summary = df_exp1.groupby(['n', 'm']).agg(
    count=('seed', 'count'),
    brd_pne_rate=('brd_found_pne', 'mean'),
    zr_opt_rate=('zr_status', lambda x: (x == 'OPTIMAL').mean()),
    avg_zr_time=('zr_time', 'mean'),
    avg_pos=('pos', 'mean'),
).round(4)
print(summary)

## 3. Experiment 2: TAS for Non-PNE Instances

In [ ]:
# ── Load Exp1 results from CSV (skip re-running Exp1) ──────────────
# Run this cell instead of cells 7-8 if you already have results.

df_exp1 = pd.read_csv(exp1_path)
for col in ['n', 'm', 'w_bar', 'u_bar', 'seed']:
    df_exp1[col] = df_exp1[col].astype(int)
df_exp1['brd_found_pne'] = df_exp1['brd_found_pne'].astype(bool)
results_exp1 = df_exp1.to_dict(orient='records')
print(f'Loaded Exp1 from {exp1_path}: {len(df_exp1)} rows')
print(f'ZR status counts:\n{df_exp1["zr_status"].value_counts().to_string()}')

In [ ]:
exp2_header = [
    'seed', 'n', 'm', 'w_bar', 'u_bar', 'delta', 'regular',
    'alpha_init', 'alpha_star',
    'alpha_ub_verified', 'alpha_lb_unverified', 'alpha_lb_verified',
    'trivial_from_heuristic', 'verified_through_binary_search',
    'n_bisect_iters',
    'tas_time_total',
]

# Gather failed instances from Exp1
failed_rows = [
    r for r in results_exp1
    if r.get('zr_status') in ('INFEASIBLE', 'TIME_LIMIT', 'INTERRUPTED', 'UNKNOWN')
    and not r.get('brd_found_pne', False)
]
print(f'Exp2 candidates: {len(failed_rows)} instances')

In [ ]:
exp2_header = [
    'seed', 'n', 'm', 'w_bar', 'u_bar', 'delta', 'regular',
    'alpha_init', 'alpha_star',
    'alpha_ub_verified', 'alpha_lb_unverified', 'alpha_lb_verified',
    'trivial_from_heuristic', 'verified_through_binary_search',
    'n_bisect_iters',
    'tas_time_total',
]

# Gather failed instances from Exp1
failed_rows = [
    r for r in results_exp1
    if r.get('zr_status') in ('INFEASIBLE', 'TIME_LIMIT', 'INTERRUPTED', 'UNKNOWN')
    and not r.get('brd_found_pne', False)
]
print(f'Exp2 candidates: {len(failed_rows)} instances')

results_exp2 = []

for count, exp1_row in enumerate(failed_rows):
    n = exp1_row['n']
    m = exp1_row['m']
    w_bar = exp1_row['w_bar']
    u_bar = exp1_row['u_bar']
    seed = exp1_row['seed']
    tag = f'n{n}_m{m}_wb{w_bar}_ub{u_bar}_d{DELTA}_s{seed}'

    try:
        inst = generate_nonregular_instance(
            n_players=n, m_bins=m,
            w_bar=w_bar, u_bar=u_bar,
            delta=DELTA, seed=seed,
        )
        regular = is_regular_instance(inst)

        tas = solve_abs(
            inst,
            eps=TAS_EPS,
            time_limit_per_call=TAS_TIME_LIMIT,
            regular=regular,
            log_to_console=0,
        )

        row2 = {
            'seed': seed, 'n': n, 'm': m,
            'w_bar': w_bar, 'u_bar': u_bar, 'delta': DELTA,
            'regular': regular,
            'alpha_init': tas.get('alpha_init'),
            'alpha_star': tas.get('alpha_star'),
            'alpha_ub_verified': tas.get('alpha_ub_verified'),
            'alpha_lb_unverified': tas.get('alpha_lb_unverified'),
            'alpha_lb_verified': tas.get('alpha_lb_verified'),
            'trivial_from_heuristic': bool(tas.get('trivial_from_heuristic')),
            'verified_through_binary_search': bool(tas.get('verified_through_binary_search')),
            'n_bisect_iters': int(tas.get('n_bisect_iters', 0)),
            'tas_time_total': float(tas.get('time_total', 0.0)),
        }
        results_exp2.append(row2)
        append_row(exp2_path, exp2_header, row2)

        print(f'[{count+1}/{len(failed_rows)}] {tag}: '
              f'alpha*={row2["alpha_star"]:.4f} '
              f'(lb_v={row2["alpha_lb_verified"]}, lb_u={row2["alpha_lb_unverified"]}) '
              f'{row2["n_bisect_iters"]} iters, {row2["tas_time_total"]:.1f}s')

    except Exception as e:
        append_row(err_path, err_header, {
            'seed': seed, 'n': n, 'm': m, 'w_bar': w_bar, 'u_bar': u_bar,
            'delta': DELTA, 'stage': 'EXP2', 'error': repr(e),
            'traceback': traceback.format_exc(),
        })
        print(f'  !! EXP2 failed ({tag}): {e}')
        continue

df_exp2 = pd.DataFrame(results_exp2)
if len(df_exp2) > 0:
    print(f'\nExp2 complete: {len(df_exp2)} rows saved to {exp2_path}')
else:
    print('\nNo instances needed TAS (all solved in Exp1).')

In [ ]:
# Experiment 2 Summary
print('=== Experiment 2: TAS Summary (Non-Regular) ===')
if len(df_exp2) > 0:
    print(f'Instances in TAS: {len(df_exp2)}')
    print(f'Mean alpha*: {df_exp2["alpha_star"].mean():.4f}')
    print(f'Max  alpha*: {df_exp2["alpha_star"].max():.4f}')
    n_verified_lb = df_exp2['alpha_lb_verified'].notna().sum()
    print(f'Verified lower bound: {n_verified_lb} ({n_verified_lb/len(df_exp2):.1%})')
    print(f'Mean bisection iters: {df_exp2["n_bisect_iters"].mean():.1f}')

    print('\n--- By (n, m) ---')
    summary2 = df_exp2.groupby(['n', 'm']).agg(
        count=('seed', 'count'),
        avg_alpha_star=('alpha_star', 'mean'),
        max_alpha_star=('alpha_star', 'max'),
        avg_iters=('n_bisect_iters', 'mean'),
    ).round(4)
    print(summary2)
else:
    print('No instances required TAS.')

## 4. Combined Results

In [ ]:
# Merge Exp1 and Exp2
df_exp1['tag'] = df_exp1.apply(
    lambda r: f"n{int(r['n'])}_m{int(r['m'])}_wb{int(r['w_bar'])}_ub{int(r['u_bar'])}_s{int(r['seed'])}",
    axis=1,
)

if len(df_exp2) > 0:
    df_exp2['tag'] = df_exp2.apply(
        lambda r: f"n{int(r['n'])}_m{int(r['m'])}_wb{int(r['w_bar'])}_ub{int(r['u_bar'])}_s{int(r['seed'])}",
        axis=1,
    )
    exp2_merge_cols = ['tag', 'alpha_init', 'alpha_star', 'alpha_lb_verified',
                       'alpha_lb_unverified', 'alpha_ub_verified', 'n_bisect_iters',
                       'tas_time_total']
    df_all = pd.merge(df_exp1, df_exp2[exp2_merge_cols], on='tag', how='left')
else:
    df_all = df_exp1.copy()
    for col in ['alpha_init', 'alpha_star', 'alpha_lb_verified',
                'alpha_lb_unverified', 'alpha_ub_verified', 'n_bisect_iters',
                'tas_time_total']:
        df_all[col] = np.nan

# For instances solved in Exp1, alpha_star = 1.0
mask_solved = (
    (df_all['zr_status'] == 'OPTIMAL')
    | (df_all['brd_found_pne'] == True)
)
df_all.loc[mask_solved & df_all['alpha_star'].isna(), 'alpha_star'] = 1.0

df_all.to_csv(combined_path, index=False)
print(f'Combined results: {len(df_all)} rows -> {combined_path}')
print(f'Columns: {list(df_all.columns)}')

# Final summary
print('\n=== Final Summary (Non-Regular) ===')
for n_val in sorted(df_all['n'].unique()):
    sub = df_all[df_all['n'] == n_val]
    n_pne = ((sub['zr_status'] == 'OPTIMAL') | (sub['brd_found_pne'] == True)).sum()
    pos_ok = sub['pos'].notna()
    pos_str = f'POS mean={sub.loc[pos_ok, "pos"].mean():.3f}' if pos_ok.any() else 'no POS'
    tas_sub = sub[sub['n_bisect_iters'].notna() & (sub['n_bisect_iters'] > 0)]
    tas_str = f'TAS: {len(tas_sub)} inst' if len(tas_sub) > 0 else 'no TAS'
    print(f'  n={n_val}: {len(sub)} inst, PNE={n_pne}, {pos_str}, {tas_str}')